#### **Python Asyncio**

`asyncio` is Python’s library for running **asynchronous code**, especially when a program spends time waiting for things like network responses, files, or timers. While one task is waiting, the event loop can let another task make progress.

**Key ideas**
- **Coroutine:** An asynchronous function declared with `async def`.
- **`await`:** Pauses a coroutine while it waits for another asynchronous operation.
- **Event loop:** Schedules and runs coroutines.
- **`asyncio.run()`:** Starts the event loop and runs an async program.
- **`asyncio.gather()`:** runs multiple awaitable tasks concurrently and waits until all of them finish. It returns their results in the same order you passed them in.



Both greetings run concurrently: while each waits for its one-second timer, the other can proceed.

##### When to use it

`asyncio` is useful when your program handles many **I/O-bound** tasks, such as making web requests or communicating with network services. It does **not** automatically make CPU-heavy work faster, and ordinary blocking calls can prevent other async tasks from running until they finish.

In [5]:

async def greet(name):
    print(f"Hello, {name}!")
    await asyncio.sleep(1)  # Wait without blocking other async tasks
    print(f"Goodbye, {name}!")

async def main():
    await asyncio.gather(
        greet("Alex"),
        greet("Sam"),
    )

# asyncio.run(main()) --> throws error becoz Jupyter already has an event loop running, and asyncio.run() tries to start another one.
await main()

Hello, Alex!
Hello, Sam!
Goodbye, Alex!
Goodbye, Sam!


### **Pydantic**

Python type hints describe the expected types, but Python does not enforce them automatically at runtime.

```python
def chat(message: str, temperature: float):
    return f"Message: {message}, Temp: {temperature}"

# Python allows this call, even though the arguments don't match the hints.
chat(message=123, temperature="high")
```

Pydantic bridges this gap by reading type annotations dynamically at runtime using `'pydantic_core'` (written in Rust) to enforce types when objects are created.

Why it exists? : When raw external data arrives (e.g., from an HTTP request, JSON file, or LLM payload), it must be both parsed into Python native objects and
validated against domain rules.

Pydantic collapses parsing and validation into one model class, executed automatically upon instantiation.

`BaseModel` is the parent class for creating data models in Pydantic. It checks that data matches the types you define and lets you access it as attributes. Makes the classes type safe

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

user = User(name="Ava", age="25")
print(user.age)  # 25 -> pydantic cn type-cast.


25


**`Validation`** - if you pass a value that cannot be type-cast, pydantic will raise a validation error.

In [2]:
from pydantic import BaseModel, ValidationError

class ChatRequest(BaseModel):
    message: str
    temperature: float
    model: str

# Automatic type coercion when compatible (e.g., string "0.7" converted to float 0.7)
valid_req = ChatRequest(message="Explain vector databases", temperature="0.7", model="gpt-40")
print(f"Valid request object created: {valid_req}")
print(f"Temperature type is float: {type(valid_req.temperature) }")

# Automatic validation error on incompatible data types
try:
    ChatRequest(message=123, temperature="invalid_number", model="gpt-4o")
except ValidationError as e:
    print("\n-Caught expected Pydantic ValidationError --- ")
    print(e)

Valid request object created: message='Explain vector databases' temperature=0.7 model='gpt-40'
Temperature type is float: <class 'float'>

-Caught expected Pydantic ValidationError --- 
2 validation errors for ChatRequest
message
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
temperature
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='invalid_number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing


**Contrained Validation & Aliases**

The `Field()` function defines constraints, default values, descriptions, and custom input/output field names (aliases).

| Constraint parameter | Description | Example |
|---|---|---|
| `min_length` / `max_length` | Restricts the length of a string | `Field(min_length=1, max_length=100)` |
| `gt` / `ge` | Requires a value to be greater than / greater than or equal to a limit | `Field(ge=0.0)` |
| `lt` / `le` | Requires a value to be less than / less than or equal to a limit | `Field(le=2.0)` |
| `alias` | Maps an external JSON key to a Python-friendly field name | `Field(alias="api_key")` |


`Annotated` lets you attach extra metadata to a type without changing the type itself.

Use `Annotated` when you want to keep a type and its related metadata together—for example, validation rules, descriptions, or API documentation. Use Annotated from typing_extensions.

In [4]:
from pydantic import Field
from typing_extensions import Annotated

class ConstrainedChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=5000)
    temperature: Annotated[float, Field(default=0.7, ge=0.0, le=2.0)]
    # Alias maps incoming key 'user_id' to internal variable name 'user_identifier'
    user_identifier: str = Field(alias="user_id")

# Passing payload using the alias name 'user_id'
req = ConstrainedChatRequest(message="Hello", user_id="usr_12345", temperature=1.4)
print(f"Parsed with Alias -> Internal Attribute: {req}")

Parsed with Alias -> Internal Attribute: message='Hello' temperature=1.4 user_identifier='usr_12345'


`@field_validator` lets you add custom validation logic to a model field. Use it when built-in constraints—such as `Field(ge=0)`—aren’t enough.

By default, the validator runs after Pydantic has converted the input to the field’s declared type. You can use `mode="before"` to validate or transform the raw input first.

`@model_validator` adds custom validation that applies to the **model as a whole**, rather than to one field. Use it when validation depends on multiple fields or the raw input object.


In [6]:
from pydantic import field_validator, model_validator

class SignupRequest(BaseModel):
    username: str
    password: str
    confirm_password: str

    # Single Field Transformation / Validation
    @field_validator("username")
    @classmethod
    def validate_username(cls, value: str) -> str:
        if " " in value:
            raise ValueError("Username cannot contain spaces")
        return value.lower() # Normalizes username to lowercase

    # Cross-Field Validation
    @model_validator(mode="after")
    def verify_passwords_match(self):
        if self.password != self.confirm_password:
            raise ValueError("Passwords do not match")
        return self

signup = SignupRequest(username="Mahima_Bhagwat", password="123", confirm_password="123")

print(signup)

username='mahima_bhagwat' password='123' confirm_password='123'


- `ConfigDict`: Its a configuration object used to define behavior for a Pydantic model. It lets you enable/disable settings like strict validation, extra-field handling, stripping whitespace, etc.
- `SecretStr`: Used for sensitive values like passwords or tokens.
It stores the value securely and hides it when displayed in logs or repr.
- `EmailStr`: A string type that validates whether the input is a real email format.


In [5]:
from pydantic import ConfigDict, SecretStr, EmailStr

class APIConfiguration(BaseModel):
# Strict configuration: throw error if extra unexpected keys are sent
    model_config = ConfigDict(extra="forbid")

    admin_email: EmailStr
    api_key: SecretStr

config = APIConfiguration(
    admin_email="admin@ai-service.com",
    api_key="sk-proj-secret-token-12345"
)

print(config)
print(config.api_key.get_secret_value())  # Accessing the secret value securely

admin_email='admin@ai-service.com' api_key=SecretStr('**********')
sk-proj-secret-token-12345


**Pydantic Serialization**
Pydantic standardizes data transformations across Python objects, dictionaries, and JSON strings.


- `model_validate()` converts data from a dictionary/DB row into a Pydantic model
- `model_dump()` converts a Pydantic model instance into a Python dictionary
- `model_dump_json()` converts a Pydantic model instance into a JSON string

In [ ]:
raw_dict = {"name": "Mahima", "age": 21, "email": "mahima@example.com"}

class User(BaseModel) :
    name: str
    age: int
    email: EmailStr

# user = User( ** raw_dict)
user = User.model_validate(raw_dict)

user_dict = user.model_dump()

user_json = user.model_dump_json()

print(user)
print(type(user_dict))

print(user_json)

name='Mahima' age=21 email='mahima@example.com'
name='Mahima' age=21 email='mahima@example.com'
<class 'dict'>
{"name":"Mahima","age":21,"email":"mahima@example.com"}
